# VTON – spúšťač na Kaggle

Tento notebook iba stiahne najnovší kód z GitHubu a spustí aplikáciu. **Kód sa neupravuje tu, ale v repe.**

**Pred prvým spustením** (vpravo v *Settings*):
- **Accelerator:** GPU (T4 x2 alebo P100)
- **Internet:** On (vyžaduje overené telefónne číslo v Kaggle účte)
- *(len pre súkromné repo)* *Add-ons → Secrets* → pridaj `GITHUB_TOKEN` a zapni ho pre tento notebook

Potom stačí **Run All**. Na konci sa vypíše verejný odkaz `https://….gradio.live`.

In [ ]:
# Nastavenia
REPO_URL = "https://github.com/MatejGalik/Semestral-VTON-work.git"
BRANCH = "main"              # vetva, ktorú chceš skúšať
CATVTON_URL = "https://github.com/Zheng-Chong/CatVTON.git"
PREFETCH_WEIGHTS = True      # stiahnuť váhy vopred (chyby sa ukážu hneď)

WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/Semestral-VTON-work"
CATVTON_DIR = f"{WORK}/CatVTON"

In [ ]:
# 1. Kontrola GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU nie je zapnuté – Settings → Accelerator → GPU"

In [ ]:
# 2. Najnovší kód projektu (pri opakovanom spustení sa len aktualizuje)
import os, subprocess

url = REPO_URL
try:  # súkromné repo: token z Kaggle Secrets
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass  # verejné repo alebo secret nie je nastavený

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, url, REPO_DIR], check=True)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 3. CatVTON (len kód, nie je súčasťou nášho repa)
if not os.path.isdir(CATVTON_DIR):
    subprocess.run(["git", "clone", "--depth", "1", CATVTON_URL, CATVTON_DIR], check=True)
os.environ["CATVTON_DIR"] = CATVTON_DIR
print("CATVTON_DIR =", CATVTON_DIR)

In [ ]:
# 4. Závislosti (torch je na Kaggle predinštalovaný)
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!if [ -f requirements-gpu.txt ]; then pip install -q -r requirements-gpu.txt; else echo "requirements-gpu.txt zatiaľ neexistuje – preskakujem"; fi

In [ ]:
# 5. Váhy z Hugging Face (ukladajú sa do cache, pri ďalšom behu v tej istej relácii sa nesťahujú znova)
if PREFETCH_WEIGHTS:
    from huggingface_hub import snapshot_download
    snapshot_download("zhengchong/CatVTON", allow_patterns=["mix-48k-1024/*", "SCHP/*", "DensePose/*"])
    print("Váhy CatVTON pripravené.")

In [ ]:
# 6. Spustenie aplikácie – počkaj na riadok s „gradio.live“ a otvor odkaz
# (bunka beží, kým ju nezastavíš; ak backend zlyhá, aplikácia beží v DEMO režime – pozri výpis vyššie)
!python app.py --share